In [1]:
%load_ext autoreload
%autoreload 2

In [2]:
from pathlib import Path

import pandas as pd

from magneton.utils import get_data_dir

In [3]:
labels_dir = get_data_dir() / "interpro_103.0" / "labels"

In [15]:
all_counts = []
for label_dir in labels_dir.iterdir():
    counts = []
    for label_set in label_dir.iterdir():
        num_labels = len(pd.read_table(label_set))
        label_type = label_set.stem.replace(".labels", "")
        counts.append((label_type, num_labels))
    counts = pd.DataFrame(counts, columns=["interpro_type", "count"]).assign(
        label_set=label_dir.stem
    )
    all_counts.append(counts)

In [20]:
all_counts = pd.concat(all_counts)

In [22]:
df = pd.pivot_table(data=all_counts, index="interpro_type", columns="label_set")
df

count                                           \
label_set              full_set selected_subset selected_subset_cutoff10   
interpro_type                                                              
Active_site               133.0            82.0                    127.0   
Binding_site               76.0            48.0                     74.0   
Conserved_site            748.0           356.0                    691.0   
Domain                  15868.0           917.0                   3506.0   
Family                  26322.0          1353.0                   6524.0   
Homologous_superfamily   3511.0          1133.0                   2159.0   
PTM                        17.0             6.0                     13.0   
Repeat                    374.0             NaN                      NaN   

                                                 
label_set              selected_subset_cutoff25  
interpro_type                                    
Active_site                               114.0  
Binding_site                               72.0  
Conserved_site                            573.0  
Domain                                   1964.0  
Family                                   3400.0  
Homologous_superfamily                   1685.0  
PTM                                         9.0  
Repeat                                      NaN

In [28]:
df.columns = df.columns.droplevel()
df

label_set,full_set,selected_subset,selected_subset_cutoff10,selected_subset_cutoff25
interpro_type,,,,
Active_site,133.0,82.0,127.0,114.0
Binding_site,76.0,48.0,74.0,72.0
Conserved_site,748.0,356.0,691.0,573.0
Domain,15868.0,917.0,3506.0,1964.0
Family,26322.0,1353.0,6524.0,3400.0
Homologous_superfamily,3511.0,1133.0,2159.0,1685.0
PTM,17.0,6.0,13.0,9.0
Repeat,374.0,NaN,NaN,NaN


In [34]:
row_order = [
    "Homologous_superfamily",
    "Domain",
    "Conserved_site",
    "Binding_site",
    "Active_site",
]
col_order = [
    "full_set",
    "selected_subset",
    "selected_subset_cutoff25",
    "selected_subset_cutoff10",
]
for_display = (
    df.loc[row_order, col_order]
    .astype(int)
    .rename(
        columns={
            "full_set": "Full SwissProt set",
            "selected_subset": "Count >= 75",
            "selected_subset_cutoff25": "Count >= 25",
            "selected_subset_cutoff10": "Count >= 10",
        },
        index=lambda x: x.replace("_", " "),
    )
)
for_display

label_set,Full SwissProt set,Count >= 75,Count >= 25,Count >= 10
interpro_type,,,,
Homologous superfamily,3511,1133,1685,2159
Domain,15868,917,1964,3506
Conserved site,748,356,573,691
Binding site,76,48,72,74
Active site,133,82,114,127


In [2]:
import os

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns

In [3]:
import bz2
import logging
import pickle
import tempfile
from functools import partial

import fire
import requests
from magneton.types import Protein

from magneton.io.internal import (
    parse_from_pkl,
    process_sharded_proteins,
)
from magneton.io.mmcif import mmcif_to_secondary_structs


In [34]:
# Replace with your path to the magneton-data huggingface dataset
data_path = Path("/weka/scratch/weka/kellislab/rcalef/data/magneton-data")


fasta_path = data_path / "sequences" / "uniprot_sprot.fasta.gz"

In [36]:
pkl_path = (
    data_path
    / "interpro_103.0"
    / "seq_splits"
    / "train_sharded"
    / "swissprot.with_ss.train.0.pkl.bz2"
)
pkl_path = Path(
    "/weka/scratch/weka/kellislab/rcalef/data/interpro/103.0/parsed_pickles/sharded_proteins.0.pkl.bz2"
)

example_prots = []
for i, prot in enumerate(parse_from_pkl(pkl_path)):
    example_prots.append(prot)
    if i == 10:
        break

# The `print()` method just pretty prints the object.
example_prots[0].print()

Protein(uniprot_id='A0A000',
        kb_id='sp|A0A000|A0A000_STRVD',
        name='A0A000_STRVD',
        length=394,
        parsed_entries=6,
        total_entries=7,
        entries=[InterproEntry(id='IPR015421',
                               element_type='Homologous_superfamily',
                               match_id='G3DSA:3.40.640.10',
                               element_name='Pyridoxal phosphate-dependent '
                                            'transferase, major domain',
                               representative=False,
                               positions=[(48, 288)]),
                 InterproEntry(id='IPR015422',
                               element_type='Homologous_superfamily',
                               match_id='G3DSA:3.90.1150.10',
                               element_name='Pyridoxal phosphate-dependent '
                                            'transferase, small domain',
                               representative=False,
             

In [37]:
AFDB_TMPL = "https://alphafold.ebi.ac.uk/files/AF-%s-F1-model_v4.cif"


def download_one_afdb_file(
    fh,
    uniprot_id: str,
) -> bool:
    url = AFDB_TMPL % uniprot_id

    try:
        r = requests.get(url, timeout=20)
        if r.status_code == 200:
            print("got content")
            fh.write(r.content)
        else:
            return False
    except Exception as e:
        print(f"downloading afdb: {e}")
        # logger.debug(f"failed to fetch AFDB file ({path}): {e}")
        return False
    return True

In [38]:
def add_secondary_structs_to_protein(
    prot: Protein,
) -> tuple[Protein, bool]:
    with tempfile.NamedTemporaryFile(suffix=".cif") as fh:
        try:
            download_one_afdb_file(fh, prot.uniprot_id)
            prot.secondary_structs = mmcif_to_secondary_structs(
                fh.name, expected_len=prot.length
            )
        except Exception as e:
            print(e)
            return prot, False
    return prot, True

In [39]:
got, worked = add_secondary_structs_to_protein(example_prots[0])
worked

got content


True

In [40]:
got.secondary_structs

[SecondaryStructure(dssp_type=<DsspType.H: 0>, start=2, end=13),
 SecondaryStructure(dssp_type=<DsspType.T: 6>, start=13, end=15),
 SecondaryStructure(dssp_type=<DsspType.P: 5>, start=15, end=21),
 SecondaryStructure(dssp_type=<DsspType.E: 2>, start=22, end=24),
 SecondaryStructure(dssp_type=<DsspType.T: 6>, start=25, end=27),
 SecondaryStructure(dssp_type=<DsspType.S: 7>, start=27, end=28),
 SecondaryStructure(dssp_type=<DsspType.T: 6>, start=28, end=30),
 SecondaryStructure(dssp_type=<DsspType.E: 2>, start=30, end=34),
 SecondaryStructure(dssp_type=<DsspType.T: 6>, start=34, end=36),
 SecondaryStructure(dssp_type=<DsspType.E: 2>, start=36, end=41),
 SecondaryStructure(dssp_type=<DsspType.S: 7>, start=41, end=43),
 SecondaryStructure(dssp_type=<DsspType.S: 7>, start=44, end=45),
 SecondaryStructure(dssp_type=<DsspType.T: 6>, start=46, end=48),
 SecondaryStructure(dssp_type=<DsspType.G: 3>, start=49, end=52),
 SecondaryStructure(dssp_type=<DsspType.H: 0>, start=53, end=67),
 SecondaryS

In [14]:
check = tempfile.NamedTemporaryFile(suffix=".cif.gz")
check

In [16]:
check.close()